<a href="https://colab.research.google.com/github/CherifNexora/CherifNexora/blob/main/juriste_first.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

instalation des depences


In [ ]:
!pip install -q gradio langchain langchain-community pypdf sentence-transformers faiss-cpu
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

Téléchargement du modèle optimisé mistral 7B

In [ ]:
import os, urllib.request

MODEL_URL = "https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf"
MODEL_PATH = "/content/mistral.gguf"

if not os.path.exists(MODEL_PATH):
    print("⬇️ Téléchargement du modèle Mistral 7B (4,1 Go)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("✅ Modèle prêt.")
else:
    print("✅ Modèle déjà présent.")

⬇️ Téléchargement du modèle Mistral 7B (4,1 Go)...
✅ Modèle prêt.


Initialisation des composants avec paramètres anti‑hallucination

In [ ]:
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer
import torch

# ----- 1. LLM local avec streaming et paramètres stricts -----
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,               # vous pouvez monter à 8192 si besoin
    n_gpu_layers=-1,          # tout sur le GPU
    n_threads=4,
    temperature=0.0,          # 0 = déterministe, aucune créativité parasite
    top_p=0.5,
    repeat_penalty=1.1,
    verbose=False,
    seed=42                   # reproductible
)
print("✅ LLM Mistral chargé")

# ----- 2. Embeddings BGE-M3 (nettement supérieurs pour l’arabe/le français) -----
embedding_model = SentenceTransformer("BAAI/bge-m3")
print("✅ Embeddings BGE-M3 chargés")

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ LLM Mistral chargé


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embeddings BGE-M3 chargés


Chargement du PDF et création de la base vectorielle

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Variable globale
db = None
SIMILARITY_THRESHOLD = 0.5   # en dessous de ce score, on refuse de répondre

def load_pdf(pdf_file):
    global db
    loader = PyPDFLoader(pdf_file.name)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    docs = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
    db = FAISS.from_documents(docs, embeddings)

    return f"✅ Document chargé — {len(docs)} extraits indexés (seuil de pertinence : {SIMILARITY_THRESHOLD})."

Assistant avec streaming et blocage des hallucinations

In [ ]:
import gradio as gr

def generate_response(question, history):
    global db
    if db is None:
        history.append((question, "⚠️ Veuillez d’abord charger un document juridique."))
        return history

    # 1. Recherche avec score de similarité
    docs_with_scores = db.similarity_search_with_score(question, k=5)
    # Filtre sur le seuil
    relevant_docs = [doc for doc, score in docs_with_scores if score >= SIMILARITY_THRESHOLD]

    if not relevant_docs:
        history.append((question, "❌ Aucune information pertinente trouvée dans le document."))
        return history

    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # 2. Prompt ultra-directif (empêche l’invention)
    system = (
        "Tu es un assistant juridique algérien. Tu réponds EXCLUSIVEMENT avec les informations du CONTEXTE ci-dessous.\n"
        "Interdiction absolue d’inventer ou d’ajouter des connaissances extérieures.\n"
        "Si la réponse n'est pas dans le contexte, dis exactement : 'المعلومة غير متوفرة في الوثيقة / Information non trouvée dans le document.'"
    )
    full_prompt = f"<s>[INST] {system}\n\nCONTEXTE :\n{context}\n\nQUESTION : {question} [/INST]"

    # 3. Génération streamée (l’historique se met à jour mot par mot)
    history.append((question, ""))
    for chunk in llm(full_prompt, max_tokens=512, stop=["</s>"], stream=True):
        token = chunk["choices"][0]["text"]
        history[-1] = (question, history[-1][1] + token)
        yield history

Interface Gradio améliorée

In [ ]:
theme = gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")

with gr.Blocks(theme=theme, title="⚖️ Assistant Juridique Algérien") as demo:

    gr.Markdown("""
    # ⚖️ Assistant Juridique Algérien — Anti‑hallucinations
    **Chargez un PDF juridique (Code, Constitution…) et posez vos questions en arabe ou français.
    Les réponses proviennent uniquement du document.**
    """)

    with gr.Row():
        pdf_input = gr.File(label="📄 Choisir un PDF", file_types=[".pdf"])
        load_btn = gr.Button("📥 Charger le document", variant="primary")
    status = gr.Markdown("")

    chatbot = gr.Chatbot(
        label="💬 Discussion",
        height=500,
        avatar_images=("🧑‍💼", "🤖"),
        show_copy_button=True,
        bubble_full_width=False
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Posez votre question (ex : ما هي الحقوق الأساسية ؟)",
            show_label=False,
            container=False,
            scale=8
        )
        send_btn = gr.Button("➤ Envoyer", variant="primary", scale=1)

    clear_btn = gr.Button("🗑️ Nouvelle discussion")

    load_btn.click(fn=load_pdf, inputs=pdf_input, outputs=status)

    msg.submit(fn=generate_response, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)
    send_btn.click(fn=generate_response, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)

    clear_btn.click(lambda: [], None, chatbot, queue=False)

/tmp/ipykernel_41752/237629072.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, title="⚖️ Assistant Juridique Algérien") as demo:
/tmp/ipykernel_41752/237629072.py:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_41752/237629072.py:16: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_41752/237629072.py:16: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.


Lancement

In [ ]:
demo.queue().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://024a1647edca7fbccd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
